Initial Setup & Imports

In [42]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

Step 1: Define Pune Official Holidays (2024-2025)

In [43]:
# Pune office holidays for 2024 and 2025
pune_holidays_2024 = [
    '2024-01-01', # New Year
    '2024-01-26', # Republic Day
    '2024-03-25', # Holi
    '2024-03-29', # Good Friday
    '2024-04-09', # Gudi Padava
    '2024-05-01', # Maharashtra Day
    '2024-08-15', # Independence Day
    '2024-09-17', # Anant Chaturdashi
    '2024-10-02', # Gandhi Jayanti
    '2024-11-01', # Diwali - Laxmi Pujan
    '2024-12-25', # Christmas
]

pune_holidays_2025 = [
    '2025-01-01', # New Year
    '2025-01-26', # Republic Day
    '2025-03-14', # Holi
    '2025-03-31', # Ramzan ID
    '2025-04-18', # Good Friday
    '2025-05-01', # Maharashtra Day
    '2025-08-15', # Independence Day
    '2025-08-27', # Anant Chaturdashi
    '2025-10-02', # Gandhi Jayanti
    '2025-10-21', # Diwali - Laxmi Pujan
    '2025-10-22', # Diwali - Padwa
    '2025-12-25', # Christmas
]

# Combine all holidays
all_holidays = pune_holidays_2024 + pune_holidays_2025
all_holidays = [pd.to_datetime(date) for date in all_holidays]

print(f"Total holidays defined: {len(all_holidays)}")
print(f"2024 holidays: {len(pune_holidays_2024)}")
print(f"2025 holidays: {len(pune_holidays_2025)}")

Total holidays defined: 23
2024 holidays: 11
2025 holidays: 12


Step 2: Create Date Range and Base Dataframe

In [44]:
# Generate date range from Jan 1, 2024 to Dec 31, 2025
start_date = '2024-01-01'
end_date = '2025-12-31'

date_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Create base dataframe
df = pd.DataFrame({
    'date': date_range
})

# Extract temporal features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek # Monday=0, Sunday=6
df['day_name'] = df['date'].dt.day_name()
df['week_of_year'] = df['date'].dt.isocalendar().week
df['quarter'] = df['date'].dt.quarter

# Mark weekends
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Mark holidays
df['is_holiday'] = df['date'].isin(all_holidays).astype(int)

print(f"Dataset created with {len(df)} days")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nWeekends: {df['is_weekend'].sum()} days")
print(f"Holidays: {df['is_holiday'].sum()} days")
df.head(10)

Dataset created with 731 days
Date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00

Weekends: 208 days
Holidays: 23 days


,date,year,month,day,day_of_week,day_name,week_of_year,quarter,is_weekend,is_holiday
0,2024-01-01,2024,1,1,0,Monday,1,1,0,1
1,2024-01-02,2024,1,2,1,Tuesday,1,1,0,0
2,2024-01-03,2024,1,3,2,Wednesday,1,1,0,0
3,2024-01-04,2024,1,4,3,Thursday,1,1,0,0
4,2024-01-05,2024,1,5,4,Friday,1,1,0,0
5,2024-01-06,2024,1,6,5,Saturday,1,1,1,0
6,2024-01-07,2024,1,7,6,Sunday,1,1,1,0
7,2024-01-08,2024,1,8,0,Monday,2,1,0,0
8,2024-01-09,2024,1,9,1,Tuesday,2,1,0,0
9,2024-01-10,2024,1,10,2,Wednesday,2,1,0,0


Step 3: Define Base Weekly Attendance Pattern

In [45]:
# Base attendance pattern for each day of the week
# Monday=0, Tuesday=1, Wednesday=2, Thursday=3, Friday=4, Saturday=5, Sunday=6
base_attendance_pattern = {
    0: 70, # Monday - Moderate
    1: 85, # Tuesday - High (primary office day)
    2: 90, # Wednesday - Highest (peak office day)
    3: 88, # Thursday - High (primary office day)
    4: 55, # Friday - Lower
    5: 0,  # Saturday - Weekend
    6: 0   # Sunday - Weekend
}

# Apply base pattern
df['base_attendance_pct'] = df['day_of_week'].map(base_attendance_pattern)

print("Base weekly attendance pattern:")
print(df.groupby('day_name')['base_attendance_pct'].first().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
))

Base weekly attendance pattern:
day_name
Monday       70
Tuesday      85
Wednesday    90
Thursday     88
Friday       55
Saturday      0
Sunday        0
Name: base_attendance_pct, dtype: int64


Step 4: Apply Seasonal Adjustments

In [46]:
# Initialize attendance percentage with base pattern
df['attendance_pct'] = df['base_attendance_pct'].copy()

# December High Leave Season (40% reduction)
# Employees take 2-3 week rotational leaves
december_mask = (df['month'] == 12) & (df['is_weekend'] == 0) & (df['is_holiday'] == 0)
df.loc[december_mask, 'attendance_pct'] *= 0.65 # 35% reduction

# March-May Low Leave Season (3-5% boost)
# Stable operational quarter
stable_quarter_mask = (df['month'].isin([3, 4, 5])) & (df['is_weekend'] == 0) & (df['is_holiday'] == 0)
df.loc[stable_quarter_mask, 'attendance_pct'] *= 1.04 # 4% boost

print("Seasonal adjustments applied:")
print(f"December attendance reduced by ~35%")
print(f"March-May attendance boosted by ~4%")

Seasonal adjustments applied:
December attendance reduced by ~35%
March-May attendance boosted by ~4%


Step 5: Apply Long Weekend Effects

In [47]:
# Identify long weekend scenarios
long_weekend_adjustments = []

for holiday in all_holidays:
    holiday_dow = holiday.dayofweek

    # If holiday is Tuesday, reduce Monday attendance
    if holiday_dow == 1: # Tuesday
        monday_before = holiday - timedelta(days=1)
        if monday_before in df['date'].values:
            idx = df[df['date'] == monday_before].index[0]
            if df.loc[idx, 'is_weekend'] == 0 and df.loc[idx, 'is_holiday'] == 0:
                df.loc[idx, 'attendance_pct'] *= np.random.uniform(0.25, 0.35) # 65-75% reduction
                long_weekend_adjustments.append(('Monday before Tuesday holiday', monday_before))

    # If holiday is Thursday, reduce Friday attendance
    if holiday_dow == 3: # Thursday
        friday_after = holiday + timedelta(days=1)
        if friday_after in df['date'].values:
            idx = df[df['date'] == friday_after].index[0]
            if df.loc[idx, 'is_weekend'] == 0 and df.loc[idx, 'is_holiday'] == 0:
                df.loc[idx, 'attendance_pct'] *= np.random.uniform(0.25, 0.35) # 65-75% reduction
                long_weekend_adjustments.append(('Friday after Thursday holiday', friday_after))

    # If holiday is Monday, reduce Friday before
    if holiday_dow == 0: # Monday
        friday_before = holiday - timedelta(days=3)
        if friday_before in df['date'].values:
            idx = df[df['date'] == friday_before].index[0]
            if df.loc[idx, 'is_weekend'] == 0 and df.loc[idx, 'is_holiday'] == 0:
                df.loc[idx, 'attendance_pct'] *= np.random.uniform(0.30, 0.40) # 60-70% reduction
                long_weekend_adjustments.append(('Friday before Monday holiday', friday_before))

    # If holiday is Friday, reduce Thursday before
    if holiday_dow == 4: # Friday
        thursday_before = holiday - timedelta(days=1)
        if thursday_before in df['date'].values:
            idx = df[df['date'] == thursday_before].index[0]
            if df.loc[idx, 'is_weekend'] == 0 and df.loc[idx, 'is_holiday'] == 0:
                df.loc[idx, 'attendance_pct'] *= np.random.uniform(0.35, 0.45) # 55-65% reduction
                long_weekend_adjustments.append(('Thursday before Friday holiday', thursday_before))

print(f"Long weekend effects applied to {len(long_weekend_adjustments)} days")
if len(long_weekend_adjustments) > 0:
    print("\nSample long weekend adjustments:")
    for desc, date in long_weekend_adjustments[:5]:
        print(f" {desc}: {date.strftime('%Y-%m-%d')}")

Long weekend effects applied to 15 days

Sample long weekend adjustments:
 Thursday before Friday holiday: 2024-01-25
 Friday before Monday holiday: 2024-03-22
 Thursday before Friday holiday: 2024-03-28
 Monday before Tuesday holiday: 2024-04-08
 Friday after Thursday holiday: 2024-08-16


Step 6: Apply Monsoon Rain Alert Effects

In [48]:
# Step 6: Fetch monsoon precipitation data using Open-Meteo for Hinjewadi
# We'll also incorporate manually provided high-precipitation records

def fetch_precipitation(lat, lon, start, end):
    import requests
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "daily": "precipitation_sum",
        "timezone": "Asia/Kolkata"
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()
    return pd.DataFrame({
        "date": pd.to_datetime(data["daily"]["time"]),
        "precipitation_mm": data["daily"]["precipitation_sum"]
    })

# location coordinates for Hinjewadi, Pune
hinjewadi_lat, hinjewadi_lon = 18.5939, 73.7497

# fetch the full range of interest for Hinjewadi
precip_df = fetch_precipitation(hinjewadi_lat, hinjewadi_lon, "2024-01-01", "2025-12-31")

# no manual records required; rely solely on fetched real-time data
combined = precip_df.copy()

# identify rain alert candidates above 50mm
candidates = combined[combined["precipitation_mm"] > 50].copy()

# shuffle and randomly drop a few to introduce variability
candidates = candidates.sample(frac=1, random_state=42).reset_index(drop=True)

# select between 15 and 25 days if possible
if len(candidates) >= 15:
    keep_n = np.random.randint(15, min(len(candidates), 25) + 1)
    rain_alert_days = candidates.loc[:keep_n-1, "date"].copy()
else:
    # not enough candidates, use them all
    rain_alert_days = candidates["date"].copy()

# choose top 20 precipitation days
num_top = min(20, len(precip_df))
if num_top > 0:
    top20 = precip_df.nlargest(num_top, "precipitation_mm")
    top20 = top20.sample(n=num_top, random_state=42)["date"].copy()
else:
    top20 = pd.Series([], dtype='datetime64[ns]')

# union with previous alerts
rain_alert_days = pd.Series(pd.concat([rain_alert_days, top20]).unique())

# Mark rain alert days in main dataframe
if 'is_rain_alert' not in df.columns:
    df['is_rain_alert'] = 0

df.loc[df['date'].isin(rain_alert_days), 'is_rain_alert'] = 1

# Apply severe rain alert effect with random reduction (8-20%) to add noise
rain_mask = df['is_rain_alert'] == 1
reduction_factors = np.random.uniform(0.08, 0.20, size=rain_mask.sum())
df.loc[rain_mask, 'attendance_pct'] *= reduction_factors

print(f"Monsoon rain alerts applied to {len(rain_alert_days)} days");
print(f"(candidates >50mm: {len(candidates)}, top20 considered: {num_top})")
print("\nSample rain alert days from combined dataset:")
if len(rain_alert_days) > 0:
    print(combined[combined['date'].isin(rain_alert_days)].head(20))
else:
    print("No rain alert days selected.")

Monsoon rain alerts applied to 20 days
(candidates >50mm: 6, top20 considered: 20)

Sample rain alert days from combined dataset:
          date  precipitation_mm
189 2024-07-08              55.2
195 2024-07-14              86.4
198 2024-07-17              32.8
205 2024-07-24              34.1
206 2024-07-25              61.1
230 2024-08-18              29.3
236 2024-08-24              58.2
245 2024-09-02              48.2
266 2024-09-23              35.9
268 2024-09-25              38.6
510 2025-05-25              31.4
511 2025-05-26              72.5
532 2025-06-16              30.1
535 2025-06-19              39.3
561 2025-07-15              31.7
595 2025-08-18              38.7
596 2025-08-19              46.1
622 2025-09-14              39.3
626 2025-09-18              31.8
636 2025-09-28              59.0


Step 7: Add Controlled Random Noise (±3%)

In [49]:
# Add Gaussian noise to simulate real-world variance
# Only for non-weekend, non-holiday, non-rain-alert days

noise_mask = (df['is_weekend'] == 0) & (df['is_holiday'] == 0) & (df['is_rain_alert'] == 0)
noise = np.random.normal(0, 3, size=noise_mask.sum()) # Mean=0, StdDev=3

df.loc[noise_mask, 'attendance_pct'] += noise

print("Random noise (±3%) added to eligible days")

Random noise (±3%) added to eligible days


Step 8: Apply Boundary Controls and Final Adjustments

In [50]:
# Ensure attendance is within 0-100% range
df['attendance_pct'] = df['attendance_pct'].clip(0, 100)

# Force weekends to 0%
df.loc[df['is_weekend'] == 1, 'attendance_pct'] = 0

# Force holidays to 0%
df.loc[df['is_holiday'] == 1, 'attendance_pct'] = 0

# Round to 2 decimal places
df['attendance_pct'] = df['attendance_pct'].round(2)

print("Boundary controls applied")
print(f"Attendance range: {df['attendance_pct'].min():.2f}% - {df['attendance_pct'].max():.2f}%")

Boundary controls applied
Attendance range: 0.00% - 100.00%


Step 9: Calculate Breakfast Participation (90% of Attendance)

In [51]:
# Breakfast participation is 90% of office attendance
df['breakfast_pct'] = (df['attendance_pct'] * 0.9).round(2)

print("Breakfast participation calculated")
print(f"Breakfast range: {df['breakfast_pct'].min():.2f}% - {df['breakfast_pct'].max():.2f}%")
print(f"\nCorrelation between attendance and breakfast: {df['attendance_pct'].corr(df['breakfast_pct']):.4f}")

Breakfast participation calculated
Breakfast range: 0.00% - 90.00%

Correlation between attendance and breakfast: 1.0000


Step 10: Create Lag Features for Time Series Analysis

In [52]:
# Previous day attendance
df['prev_day_attendance'] = df['attendance_pct'].shift(1)

# Previous week same day attendance
df['prev_week_attendance'] = df['attendance_pct'].shift(7)

Step 11: Add Additional Features for ML

In [53]:
# Is it start of month (day 1-5)?
df['is_month_start'] = (df['day'] <= 5).astype(int)

# Is it end of month (last 5 days)?
df['is_month_end'] = (df['day'] >= 26).astype(int)

# Is it high leave season (Dec, Jan)?
df['is_high_leave_season'] = df['month'].isin([12, 1]).astype(int)

# Is it monsoon season (Jun-Sep)?
df['is_monsoon_season'] = df['month'].isin([6, 7, 8, 9]).astype(int)

# Is it stable season (Mar-May)?
df['is_stable_season'] = df['month'].isin([3, 4, 5]).astype(int)

# Days until next holiday
df['days_to_next_holiday'] = 0
for idx in df.index:
    current_date = df.loc[idx, 'date']
    future_holidays = [h for h in all_holidays if h > current_date]
    if future_holidays:
        df.loc[idx, 'days_to_next_holiday'] = (min(future_holidays) - current_date).days
    else:
        df.loc[idx, 'days_to_next_holiday'] = 365

# Days since last holiday
df['days_since_last_holiday'] = 0
for idx in df.index:
    current_date = df.loc[idx, 'date']
    past_holidays = [h for h in all_holidays if h < current_date]
    if past_holidays:
        df.loc[idx, 'days_since_last_holiday'] = (current_date - max(past_holidays)).days
    else:
        df.loc[idx, 'days_since_last_holiday'] = 365

print("Additional features created for ML modeling")

Additional features created for ML modeling


In [54]:
df.head(10)

,date,year,month,day,day_of_week,day_name,week_of_year,quarter,is_weekend,is_holiday,...,breakfast_pct,prev_day_attendance,prev_week_attendance,is_month_start,is_month_end,is_high_leave_season,is_monsoon_season,is_stable_season,days_to_next_holiday,days_since_last_holiday
0,2024-01-01,2024,1,1,0,Monday,1,1,0,1,...,0.00,NaN,NaN,1,0,1,0,0,25,365
1,2024-01-02,2024,1,2,1,Tuesday,1,1,0,0,...,74.88,0.00,NaN,1,0,1,0,0,24,1
2,2024-01-03,2024,1,3,2,Wednesday,1,1,0,0,...,83.56,83.20,NaN,1,0,1,0,0,23,2
3,2024-01-04,2024,1,4,3,Thursday,1,1,0,0,...,79.98,92.84,NaN,1,0,1,0,0,22,3
4,2024-01-05,2024,1,5,4,Friday,1,1,0,0,...,47.78,88.87,NaN,1,0,1,0,0,21,4
5,2024-01-06,2024,1,6,5,Saturday,1,1,1,0,...,0.00,53.09,NaN,0,0,1,0,0,20,5
6,2024-01-07,2024,1,7,6,Sunday,1,1,1,0,...,0.00,0.00,NaN,0,0,1,0,0,19,6
7,2024-01-08,2024,1,8,0,Monday,2,1,0,0,...,60.25,0.00,0.00,0,0,1,0,0,18,7
8,2024-01-09,2024,1,9,1,Tuesday,2,1,0,0,...,76.06,66.94,83.20,0,0,1,0,0,17,8
9,2024-01-10,2024,1,10,2,Wednesday,2,1,0,0,...,79.56,84.51,92.84,0,0,1,0,0,16,9


Step 12: Export Dataset

In [55]:
# ensure breakfast_pct column exists (compute if previous steps skipped)
if 'breakfast_pct' not in df.columns and 'attendance_pct' in df.columns:
    df['breakfast_pct'] = (df['attendance_pct'] * 0.9).round(2)

# Select basic columns that would come from canteen records
# This represents realistic data that the canteen would maintain
canteen_columns = [
    'date',            # Date of record
#    'day_name',        # Day name (for easy reference)
#    'is_weekend',      # Weekend flag
#    'is_holiday',      # Holiday flag
    'breakfast_pct',   # Daily breakfast footfall percentage (TARGET variable)
#    'is_rain_alert'    # More than 50mm rain
]

df_canteen = df[canteen_columns].copy()

# Rename columns to make it look like raw canteen data
df_canteen.columns = ['date', 'breakfast_footfall_pct']

# Export to CSV
output_file = 'canteen_breakfast_data.csv'
df_canteen.to_csv(output_file, index=False)

print(f"✔ Canteen dataset exported successfully")
print(f"File: {output_file}")
print(f"Records: {len(df_canteen)}")
print(f"Columns: {len(df_canteen.columns)}")
print(f"\nCanteen Data Columns:")
for i, col in enumerate(df_canteen.columns, 1):
    print(f" {i}. {col}")

✔ Canteen dataset exported successfully
File: canteen_breakfast_data.csv
Records: 731
Columns: 2

Canteen Data Columns:
 1. date
 2. breakfast_footfall_pct
